In [1]:
from netgen.meshing import Mesh
from ngsolve import *
from ngsolve.krylovspace import CGSolver
from netgen.occ import *
from ngsolve.webgui import Draw

import matplotlib.pylab as plt
import scipy.sparse as sp

In [ ]:
def Capacitor3DGeometry(box_size, W, L, d, D, h_max):
    air_box = Box((-box_size, -box_size, -box_size), (box_size, box_size, box_size))
    air_box.faces.name = "Outer"  

    dielectric = Box((-W/2, -d/2, -L/2), (W/2, d/2, L/2))
    dielectric.faces.maxh = h_max/2
    dielectric.name = "dielectric"

    electrode_positive = Box((-W/2, d/2, -L/2), (W/2, (d+D)/2, L/2))
    electrode_positive.faces.maxh = h_max/4
    electrode_positive.faces.name = "electrode_positive"

    electrode_negative = Box((-W/2, -(d+D)/2, -L/2), (W/2, -d/2, L/2))
    electrode_negative.faces.maxh = h_max/4
    electrode_negative.faces.name = "electrode_negative"

    air = air_box - dielectric
    air.name = "air"

    shape = Glue([air, dielectric])
    shape = shape - electrode_positive - electrode_negative

    return shape


def Capacitor3DMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=3).GenerateMesh(maxh=h_max))

    return mesh


def Capacitor3DWeakForm(mesh, FE_order, epsr):

    fes_phi = H1(mesh, order=FE_order, dirichlet="el.*")
    fes_E = HCurl(mesh, order=FE_order-1)
    fes_D = HDiv(mesh, order=FE_order-1)
    fes_rho = L2(mesh, order=FE_order-2)

    u = fes_phi.TrialFunction()
    v = fes_phi.TestFunction()

    a = BilinearForm(epsr*grad(u)*grad(v)*dx)
    #precond = preconditioners.Local(a)
    #precond = preconditioners.MultiGrid(a)
    precond = preconditioners.BDDC(a)

    return a, precond, fes_phi, fes_E, fes_D, fes_rho


def Capacitor3DAssemble(a):
     
    pre = preconditioners.Local(a)
     
    with TaskManager():
        a.Assemble()

    return a
    

def Capacitor3DSolver(mesh, fes, a, precond):

    potential_gf = GridFunction(fes)
    potential_gf.Interpolate(mesh.BoundaryCF({"electrode_positive":1, "electrode_negative":-1 }), mesh.Boundaries(".*"))
        

    with TaskManager():
        inv = CGSolver(
            mat=a.mat,
            pre=precond,
            printrates='\r',
            maxiter=10000
        )
      
        potential_gf.vec.data -= inv*(a.mat * potential_gf.vec)

    return potential_gf

In [3]:
clipping = {"function": True, "pnt": (0, 0, 0), "vec": (0, 0, -1)}

In [ ]:
box_size, W, L, d, D = 15, 5, 5, 1.5, 0.5
epsr_air, epsr_dielectric = 1.0, 4.0

FE_order = 2
h_max = 0.6

geo = Capacitor3DGeometry(box_size, W, L, d, D, h_max)

In [5]:
mesh = Capacitor3DMesh(geo, h_max)

In [6]:
epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

In [7]:
a, precond, fes_phi, fes_E, fes_D, fes_rho = Capacitor3DWeakForm(mesh, FE_order, epsr)

In [8]:
a = Capacitor3DAssemble(a)

In [9]:
phi_gf = Capacitor3DSolver(mesh, fes_phi, a, precond)

CG converged in 2 iterations to residual 8.08518519199422e-14


In [10]:
Draw (phi_gf, scale=5, clipping=clipping);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…